In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.gold.customer_metrics') (
    customer_id INT,
    customer_name STRING,
    state STRING,
    first_order_date DATE,
    last_order_date DATE,
    lifetime_value DECIMAL(14,2),
    total_orders BIGINT,
    avg_order_value DECIMAL(10,2),
    customer_segment STRING
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW customer_orders_sequenced AS
SELECT
    c.id AS customer_id,
    c.name AS customer_name,
    c.state,
    s.sale_id,
    s.sale_date,
    s.sale_amount,
    FIRST_VALUE(s.sale_date) OVER (
        PARTITION BY c.id ORDER BY s.sale_date
    ) AS first_order_date,
    LAG(s.sale_date) OVER (
        PARTITION BY c.id ORDER BY s.sale_date
    ) AS previous_order_date,
    LEAD(s.sale_date) OVER (
        PARTITION BY c.id ORDER BY s.sale_date
    ) AS next_order_date
FROM identifier(:catalog || '.silver.sales_clean') s
JOIN identifier(:catalog || '.silver.customers_clean') c ON s.customer_id = c.id;
 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW customer_clv AS
SELECT
    customer_id,
    customer_name,
    state,
    MIN(sale_date) AS first_order_date,
    MAX(sale_date) AS last_order_date,
    SUM(sale_amount) AS lifetime_value,
    COUNT(*) AS total_orders,
    ROUND(SUM(sale_amount) / COUNT(*), 2) AS avg_order_value,
    CASE
        WHEN COUNT(*) = 1 THEN 'new'
        WHEN COUNT(*) BETWEEN 2 AND 4 THEN 'returning'
        ELSE 'loyal'
    END AS customer_segment
FROM customer_orders_sequenced
GROUP BY customer_id, customer_name, state;
 

In [0]:
%sql
INSERT OVERWRITE identifier(:catalog || '.gold.customer_metrics')
SELECT customer_id, customer_name, state, first_order_date, last_order_date,
       lifetime_value, total_orders, avg_order_value, customer_segment
FROM customer_clv;

In [0]:
%sql
SELECT customer_segment,
       COUNT(*) AS num_customers,
       ROUND(AVG(avg_order_value), 2) AS segment_avg_order_value,
       ROUND(AVG(lifetime_value), 2) AS segment_avg_clv
FROM identifier(:catalog || '.gold.customer_metrics')
GROUP BY customer_segment;

In [0]:
%sql
SELECT
    date_trunc('month', sale_date) AS sale_month,
    SUM(CASE WHEN sale_date = first_order_date THEN 1 ELSE 0 END) AS new_customers,
    SUM(CASE WHEN sale_date <> first_order_date THEN 1 ELSE 0 END) AS returning_orders
FROM customer_orders_sequenced
GROUP BY date_trunc('month', sale_date)
ORDER BY sale_month;